# 01 — Mean-Variance: GMV & Tangency

The Markowitz core: the global minimum-variance portfolio and the
maximum-Sharpe (tangency) portfolio, the two long-only strategies every later
method in this repo is measured against.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import maplab as ml
from maplab import Panel, apply_style
from maplab.models import tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

## The math

**Unconstrained global minimum-variance (GMV):**

$$w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}$$

**Unconstrained tangency (maximum Sharpe):**

$$w_{\text{tan}} = \frac{\Sigma^{-1}(\mu - r_f)}{\mathbf{1}'\Sigma^{-1}(\mu - r_f)}$$

**Two-fund separation:** every unconstrained mean-variance-efficient portfolio
is a combination of just these two funds, $w_{GMV}$ and $w_{tan}$ — no other
information from $\mu$ or $\Sigma$ is needed once you have them.

**Why the classes in `maplab.models` are long-only, not these closed forms:**
the unconstrained solutions above can (and, on this universe, do) put large
long and short positions on assets with noisy estimated means — leverage and
short exposure that a real long-only allocator can't take and that is mostly
estimation error, not signal. `GMV` and `MaxSharpe` solve the same objectives
numerically with $w \ge 0$ added, which is why they need `scipy.optimize`
rather than a one-line solve.

**$\mu$ estimation (log → arithmetic):** the harness estimates $(\mu, \Sigma)$
on **log** returns (additive, well-behaved) but scores strategies on
**arithmetic** (simple) returns. Feeding a log-return mean straight into a
mean-variance objective is systematically biased downward, so:

$$\mu = \overline{r_{\log}} \times 252 + \tfrac{1}{2}\,\text{diag}(\Sigma)$$

**Sharpe:**

$$\text{Sharpe} = \frac{\text{mean}(r - r_f)}{\text{std}(r - r_f)} \times \sqrt{252}$$

where $r_f$ is BIL's daily simple return (`ml.load_rf_returns()`) — the
actual T-bill carry, not a fixed hurdle. `MaxSharpe` uses the trailing-252-day
mean of that same BIL series, annualized, as $r_f$ at each rebalance
(`rf="panel"`, the default — see `MaxSharpe._rf_ann`).

## Snapshot at the train/test split

$(\mu, \Sigma)$ estimated at the last rebalance on or before
`TRAIN_TEST_SPLIT`, and what GMV, MaxSharpe, and the **unconstrained**
tangency portfolio each do with them. The unconstrained column is the
"why long-only" evidence: its largest longs and shorts, and its gross
leverage $\sum |w|$.

In [ ]:
from maplab import GMV, MaxSharpe

rdates = ml.rebalance_dates(simple_returns.index)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
split_rebal = rdates[rdates <= split_ts].max()
print(f"Snapshot date (last rebalance <= {ml.TRAIN_TEST_SPLIT}): {split_rebal.date()}")

gmv_snap = GMV()
mu, Sigma = gmv_snap._estimate(panel, split_rebal)

rf_ann = MaxSharpe()._rf_ann(panel, split_rebal)
print(f"rf_ann at snapshot (trailing-252-day BIL mean, annualized): {rf_ann:.4%}")

w_gmv = GMV()(panel, split_rebal)
w_msr = MaxSharpe()(panel, split_rebal)

try:
    w_tan = tangency_closed_form(mu, Sigma, rf=rf_ann)
    tan_undefined = False
except ValueError as e:
    tan_undefined = True
    tan_error = str(e)

cols = {"GMV (long-only)": w_gmv, "MaxSharpe (long-only)": w_msr}
if not tan_undefined:
    cols["Tangency (UNCONSTRAINED)"] = w_tan
snap = pd.DataFrame(cols).round(4)

leverage_msg = (f"Gross leverage sum|w|  —  GMV: {w_gmv.abs().sum():.3f}  "
                f"MaxSharpe: {w_msr.abs().sum():.3f}")
if not tan_undefined:
    leverage_msg += f"  Unconstrained tangency: {w_tan.abs().sum():.3f}"
print(leverage_msg)

if tan_undefined:
    print(f"\nUnconstrained tangency is UNDEFINED at this snapshot:\n  {tan_error}")
    print("(1'Σ⁻¹(μ−rf) = (1'Σ⁻¹1)(μ_GMV,unc − rf), and 1'Σ⁻¹1 > 0, so the "
          "denominator is negative iff the unconstrained GMV's own expected "
          "return is below rf — the tangency then lies on the inefficient "
          "branch of the hyperbola, not a numerically \"extreme\" point on "
          "the efficient one.)")
else:
    print("\nUnconstrained tangency — largest LONGS:")
    display(w_tan.sort_values(ascending=False).head(5).round(4).to_frame("weight"))
    print("Unconstrained tangency — largest SHORTS:")
    display(w_tan.sort_values().head(5).round(4).to_frame("weight"))

print("\nHoldings with any weight > 1e-4:")
mask = (snap.abs() > 1e-4).any(axis=1)
snap.loc[mask].sort_values("GMV (long-only)", ascending=False)

In [ ]:
from maplab.models import gmv_closed_form

w_unc = gmv_closed_form(Sigma)
mu_gmv_unc = float(w_unc.to_numpy() @ mu.to_numpy())
ones_Sinv_ones = float(np.ones(len(mu)) @ np.linalg.solve(Sigma.to_numpy(), np.ones(len(mu))))
denom_from_identity = ones_Sinv_ones * (mu_gmv_unc - rf_ann)
raw_denom = float(np.ones(len(mu)) @ np.linalg.solve(Sigma.to_numpy(), (mu - rf_ann).to_numpy()))

print(f"mu' w_unc (unconstrained GMV's own expected return): {mu_gmv_unc:.6f}  (< rf_ann={rf_ann:.6f}? {mu_gmv_unc < rf_ann})")
print(f"1'Σ⁻¹1: {ones_Sinv_ones:.6f}")
print(f"(1'Σ⁻¹1) * (mu_GMV,unc - rf_ann): {denom_from_identity:.6f}   vs.   raw denominator 1'Σ⁻¹(μ-rf_ann): {raw_denom:.6f}")

print(f"\nUnconstrained GMV gross leverage sum|w|: {w_unc.abs().sum():.3f}")
print("\nUnconstrained GMV — largest LONGS:")
display(w_unc.sort_values(ascending=False).head(5).round(4).to_frame("weight"))
print("Unconstrained GMV — largest SHORTS:")
display(w_unc.sort_values().head(5).round(4).to_frame("weight"))

## The long-only efficient frontier

At the same snapshot date: sweep a target return, minimize variance subject
to $w'\mu = \text{target}$, $\sum w = 1$, $w \ge 0$ (SLSQP with analytic
gradients). The 13 assets are shown as labeled dots, GMV and MaxSharpe are
marked, and the capital market line (CML) starts at $(0, r_f)$ — the
snapshot's trailing-252-day BIL mean — and passes through MaxSharpe. $\mu$
here is the trailing-252-day arithmetic mean, so the tangency tilts hard
toward last year's winners — at end-2022, ~74% in the dollar (UUP).

The grey backdrop is 20,000 random long-only portfolios drawn from a
Dirichlet($\alpha$) distribution on the 13 assets. Dirichlet(1) — the
textbook choice on 6 assets (median effective N $\approx$ 3.7) — spreads out
on 13 assets (median effective N $\approx$ 7.3, median max weight $\approx$
23%) and never reaches the corners where GMV and MaxSharpe live. The next
cell sweeps $\alpha \in \{1, 0.3, 0.1\}$ on the 13-asset universe and picks
the one whose median effective N is closest to that 6-asset reference:
$\alpha=0.3$ (median effective N $\approx$ 4.1, median max weight $\approx$
38%).

In [ ]:
def effective_n_stats(n_assets: int, alpha: float, size: int = 20_000, seed: int = 7) -> dict:
    rng = np.random.default_rng(seed)
    w = rng.dirichlet(np.full(n_assets, alpha), size=size)
    eff_n = 1.0 / np.sum(w ** 2, axis=1)
    return {"alpha": alpha, "median_eff_n": float(np.median(eff_n)), "median_max_w": float(np.median(w.max(axis=1)))}

n = len(ml.UNIVERSE)
reference_eff_n = effective_n_stats(6, 1.0)["median_eff_n"]  # Dirichlet(1) on 6 assets, the textbook case

dirichlet_table = pd.DataFrame([effective_n_stats(n, a) for a in (1.0, 0.3, 0.1)]).set_index("alpha")
dirichlet_table["abs_diff_from_ref"] = (dirichlet_table["median_eff_n"] - reference_eff_n).abs()
mc_alpha = float(dirichlet_table["abs_diff_from_ref"].idxmin())

print(f"Reference: Dirichlet(1) on 6 assets — median effective N = {reference_eff_n:.3f}")
print(f"Chosen alpha on {n} assets: {mc_alpha} "
      f"(median effective N {dirichlet_table.loc[mc_alpha, 'median_eff_n']:.3f}, "
      f"median max weight {dirichlet_table.loc[mc_alpha, 'median_max_w']:.3f})")
dirichlet_table.round(4)

In [ ]:
def min_variance_at_target(Sigma: pd.DataFrame, mu: pd.Series, target: float, x0: np.ndarray) -> np.ndarray:
    n = Sigma.shape[0]
    Sigma_np = Sigma.to_numpy()
    mu_np = mu.to_numpy()
    ones = np.ones(n)
    cons = [
        {"type": "eq", "fun": lambda w: w.sum() - 1.0, "jac": lambda w: ones},
        {"type": "eq", "fun": lambda w: w @ mu_np - target, "jac": lambda w: mu_np},
    ]
    bounds = [(0.0, 1.0)] * n
    res = opt.minimize(
        lambda w: w @ Sigma_np @ w, x0, jac=lambda w: 2 * Sigma_np @ w, method="SLSQP",
        bounds=bounds, constraints=cons, options={"ftol": 1e-12, "maxiter": 1000},
    )
    if not res.success:
        return None
    return res.x

n = len(ml.UNIVERSE)
Sigma_np = Sigma.to_numpy()
mu_np = mu.to_numpy()

# Monte Carlo backdrop: 20,000 random long-only portfolios, Dirichlet(mc_alpha)
rng_mc = np.random.default_rng(7)
mc_weights = rng_mc.dirichlet(np.full(n, mc_alpha), size=20_000)
mc_ret = mc_weights @ mu_np
mc_vol = np.sqrt(np.einsum("ij,jk,ik->i", mc_weights, Sigma_np, mc_weights))

targets = np.linspace(mu.min(), mu.max(), 60)
x0 = np.full(n, 1.0 / n)
frontier_ret, frontier_vol = [], []
for t in targets:
    w = min_variance_at_target(Sigma, mu, t, x0)
    if w is None:
        continue
    x0 = w  # warm start the next point
    frontier_ret.append(w @ mu_np)
    frontier_vol.append(np.sqrt(w @ Sigma_np @ w))
frontier_ret = np.array(frontier_ret)
frontier_vol = np.array(frontier_vol)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(mc_vol, mc_ret, s=2, alpha=0.15, color="#bbbbbb", zorder=0,
           label=f"random long-only portfolios (Dirichlet α={mc_alpha})")

asset_vol = np.sqrt(np.diag(Sigma_np))
ax.scatter(asset_vol, mu_np, s=18, color="#999999", zorder=2)
for i, tkr in enumerate(ml.UNIVERSE):
    ax.annotate(tkr, (asset_vol[i], mu_np[i]), fontsize=6,
                xytext=(3, 3), textcoords="offset points", color="#666666")

gmv_ret = float(w_gmv.to_numpy() @ mu_np)
gmv_vol = float(np.sqrt(w_gmv.to_numpy() @ Sigma_np @ w_gmv.to_numpy()))
msr_ret = float(w_msr.to_numpy() @ mu_np)
msr_vol = float(np.sqrt(w_msr.to_numpy() @ Sigma_np @ w_msr.to_numpy()))

# Efficient (return >= GMV's) vs inefficient branch of the swept curve
efficient = frontier_ret >= gmv_ret
ax.plot(frontier_vol[~efficient], frontier_ret[~efficient], color="#999999",
        ls="--", lw=1.5, zorder=3, label="Inefficient branch")
ax.plot(frontier_vol[efficient], frontier_ret[efficient], color="#2e7d32",
        lw=2, zorder=3, label="Efficient frontier (long-only)")

ew_w = np.full(n, 1.0 / n)
ew_ret = float(ew_w @ mu_np)
ew_vol = float(np.sqrt(ew_w @ Sigma_np @ ew_w))
ax.scatter([ew_vol], [ew_ret], marker="D", s=90,
           color=ml.FAMILY_COLORS["Benchmark"], zorder=4, label="EqualWeight")

ax.scatter([gmv_vol], [gmv_ret], marker="*", s=220,
           color=ml.FAMILY_COLORS[GMV.family], zorder=4, label="GMV")
ax.scatter([msr_vol], [msr_ret], marker="*", s=220,
           color=ml.FAMILY_COLORS[MaxSharpe.family], zorder=4, label="MaxSharpe")

cml_x = np.linspace(0, max(asset_vol.max(), msr_vol, mc_vol.max()) * 1.05, 20)
cml_slope = (msr_ret - rf_ann) / msr_vol
ax.plot(cml_x, rf_ann + cml_slope * cml_x, color="#1f4e79", ls="--", lw=1.5,
        zorder=1, label=f"CML (rf={rf_ann:.2%} → MaxSharpe)")

ax.set_xlabel("annualized volatility")
ax.set_ylabel("annualized arithmetic expected return (μ)")
ax.set_title(f"Long-only efficient frontier — snapshot {split_rebal.date()}")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## Backtest: GMV, MaxSharpe, EqualWeight

Fresh strategy instances run through `ml.backtest` (default scoring start —
the first eligible rebalance after the 252-day warm-up). `ml.summary` over
the full sample, the train window ($\le$ `TRAIN_TEST_SPLIT`), and the held-out
test window (> split).

In [ ]:
from maplab import EqualWeight, backtest, summary

strategies = {"GMV": GMV(), "MaxSharpe": MaxSharpe(), "EqualWeight": EqualWeight()}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag}

split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    windows = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in windows.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

msr = strategies["MaxSharpe"]
print(f"MaxSharpe fallback_dates: {len(msr.fallback_dates)}   retry_dates: {len(msr.retry_dates)}")

summary_table.round(4)

## Allocation through time

Target weights from each strategy's `wlog`, aggregated by `ml.GROUP_OF` into
the 6 asset groups, at every rebalance.

In [ ]:
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = plt.cm.tab10(np.linspace(0, 1, len(group_order)))

def weights_by_group(wlog):
    grouped = wlog.T.groupby(ml.GROUP_OF).sum().T
    return grouped.reindex(columns=group_order).fillna(0.0)

gmv_group = weights_by_group(results["GMV"]["wlog"])
msr_group = weights_by_group(results["MaxSharpe"]["wlog"])

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
for ax, grouped, title in zip(axes, [gmv_group, msr_group], ["GMV", "MaxSharpe"]):
    ax.stackplot(grouped.index, grouped.T.to_numpy(), labels=group_order, colors=group_colors)
    ax.axvline(split_ts, color="black", ls=":", lw=1)
    ax.set_ylim(0, 1)
    ax.set_title(f"{title} — target weights by asset group")
axes[-1].set_xlabel("rebalance date")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({"GMV": gmv_group.mean(), "MaxSharpe": msr_group.mean()})
display(group_means.round(4))

uup_w = results["GMV"]["wlog"]["UUP"]
print(f"GMV: UUP mean weight = {uup_w.mean():.4f}  (min={uup_w.min():.4f}, max={uup_w.max():.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for name, strat in strategies.items():
    net = results[name]["net"]
    wealth = (1.0 + net).cumprod()
    color = ml.FAMILY_COLORS[strat.family]
    ax.plot(wealth.index, wealth.to_numpy(), label=strat.label, color=color, lw=1.6)

ax.axvline(split_ts, color="#888888", ls=":", lw=1, label="train/test split")
ax.set_yscale("log")
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator
ymin, ymax = ax.get_ylim()
candidate_ticks = [1, 2, 3, 4, 6, 8]
yticks = [t for t in candidate_ticks if ymin <= t <= ymax]
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
ax.yaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylabel("cumulative wealth (log scale, net of costs)")
ax.set_title("GMV vs MaxSharpe vs EqualWeight — cumulative wealth")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

**TAKEAWAY**
- **On a clean, one-premium-per-asset universe the verdict is not "mean-variance loses."** Sharpe vs BIL: EW 0.74 ≈ MaxSharpe 0.72 > GMV 0.62 (full window). The ranking flips by window: EW leads pre-2023 (0.70 vs 0.60 / 0.59), MaxSharpe leads 2023+ (1.13 vs 0.91 / 0.76). That window-dependence is the repo's thesis in miniature.
- **None of those Sharpe gaps is statistically meaningful.** The i.i.d. standard error of a Sharpe ratio is ≈ √((1 + SR²/2)/T): about 0.27 over the full 17 years, about 0.7 over the 3.3-year test window.
- **What *is* robust is the risk profile and what it costs to get there.** GMV: 3.2% vol, −6.7% max DD. MaxSharpe: 6.3% vol, −10.8%. EW: 7.9% vol, −17.8%. MaxSharpe turns over ~248%/yr, 18× EW's 14%, and its weights jump to near-100% single-group corners (equity 2014 and 2017–18, TIPS 2021, 74% dollar at end-2022). The error-maximizer shows up in the weights and the trading, not in the Sharpe ratio.
- **GMV no longer hides in cash.** With BIL as the numeraire it must choose among risk premia, and it leans on the dollar (UUP 37% on average, range 18–58%) because the dollar rallies when risk assets sell off: a diversifier, not a low-vol accident.
- **Long-only is not cosmetic.** At end-2022 the unconstrained tangency is defined but carries 13× gross leverage (+193% UUP, −216% TIPS).
- **Estimated vs realized efficiency:** ex-ante, EW sits deep inside the frontier cloud; ex-post it has the highest wealth. Sample-mean μ is the weak link. Better μ (03, Black-Litterman) and better Σ (04, shrinkage) come next. One historical path, no bootstrap confidence intervals yet (Phase 2).